# Neural-network stability and training duration

Manually train 10 models with predetermined seeds (42–51); do **not** fit a final model on all data. Each seed combines elections through 2015 with half of 2017 for training, uses the other half of 2017 for validation, and evaluates all 2019 rows only after restoring the validation-selected checkpoint.

The median best epoch is the candidate fixed training duration for a later experiment. Variability here combines random initialisation, mini-batch order, and the changing 2017 split; it does not isolate initialisation alone.

Requirements: pandas, NumPy, SciPy, scikit-learn >= 1.2, PyTorch, and a Jupyter Python kernel. Run cells in order. CPU execution is intentional for simple reproducibility; results can still differ across library versions/platforms.

This notebook implements the split, preprocessing, training loop, early stopping, and individual evaluation explicitly, without calling the automated pipeline or NeuralNetworkTrainer. Parameters match Models/NN01_model.py. The results table reports each model's best checkpoint epoch, total epochs run (stopping_epoch), and 2019 accuracy.

In [1]:
from pathlib import Path
import copy
import random
import warnings

import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display

SEEDS = tuple(range(111223, 111233))  # Same ten seeds as NN01_model.py; fixed before evaluation
BATCH_SIZE = 64
MAX_EPOCHS = 1500
PATIENCE = 20
MIN_DELTA = 0.001
LEARNING_RATE = 0.001
HIDDEN_SIZES = (64, 32)
DEVICE = torch.device("cpu")
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

print({"numpy": np.__version__, "pandas": pd.__version__,
       "scipy": scipy.__version__, "sklearn": sklearn.__version__,
       "torch": torch.__version__, "device": str(DEVICE)})

{'numpy': '2.5.3', 'pandas': '3.0.5', 'scipy': '1.18.1', 'sklearn': '1.9.1', 'torch': '2.14.0+cu130', 'device': 'cpu'}


## Read the data and define predictors

The inspected CSV uses **election** and **winner**, with classes con, lab, lib, natSW, and oth. It currently contains 632 rows in each of 2017 and 2019. Several previous-election predictors have missing values.

Use the same 14 predictor names as Models/logistic_regression.py, listed explicitly so this experiment remains readable. Unlike that model's filtering, retain missing-predictor rows and the oth class. Exclude current-election vote shares, majority, identifiers, and election year from predictors. No feature selection uses 2019 outcomes. This assumes the upstream polling and projected-share features were constructed using information available before each election.

Inspect development data below; hold 2019 aside until evaluation.

In [2]:
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "TEST_TRAIN" / "train.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the project root or a folder beneath it.")

data = pd.read_csv(PROJECT_ROOT / "TEST_TRAIN" / "train.csv")
YEAR, TARGET = "election", "winner"
FEATURES = [
    "country/region", "previous_majority_proportion", "previous_winner",
    "Conservative", "Labour", "LD", "incumbent",
    "previous_con_share", "previous_lib_share", "previous_lab_share",
    "previous_natSW_share", "projected_con_share",
    "projected_lib_share", "projected_lab_share",
]
required = [YEAR, TARGET, *FEATURES]
missing = set(required) - set(data.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

years = pd.to_numeric(data[YEAR], errors="raise")
if years.isna().any():
    raise ValueError("Election years must not be missing.")
historical = data.loc[years <= 2015].copy()
election_2017 = data.loc[years == 2017].copy()
test_2019 = data.loc[years == 2019].copy()
if historical.empty or len(election_2017) < 2 or test_2019.empty:
    raise ValueError("Need historical rows, at least two 2017 rows, and 2019 evaluation rows.")
if test_2019[TARGET].isna().any():
    raise ValueError("2019 evaluation requires known winners for every row.")
development = pd.concat([historical, election_2017])
if development[TARGET].isna().any():
    raise ValueError("Training/validation targets must not be missing.")

# Match the explicit feature groups in Models/logistic_regression.py and NN01_model.py.
categorical_columns = ["country/region", "previous_winner", "incumbent"]
numeric_columns = [c for c in FEATURES if c not in categorical_columns]
display(pd.DataFrame({
    "dtype": development[FEATURES].dtypes.astype(str),
    "missing_development": development[FEATURES].isna().sum(),
}))
display(pd.crosstab(development[YEAR], development[TARGET]))
print("Numeric:", numeric_columns)
print("Categorical:", categorical_columns)

# Match the pipeline: share a class mapping from development data, excluding 2019.
label_encoder = LabelEncoder().fit(development[TARGET])
print("Training-derived class mapping:", dict(enumerate(label_encoder.classes_)))

,dtype,missing_development
country/region,str,0
previous_majority_proportion,float64,92
previous_winner,str,92
Conservative,float64,0
Labour,float64,0
LD,float64,0
incumbent,str,0
previous_con_share,float64,96
previous_lib_share,float64,98
previous_lab_share,float64,97


winner,con,lab,lib,natSW,oth
election,,,,,
1987,376,229,22,6,0
1992,336,271,20,7,0
1997,166,418,45,10,2
2001,166,412,52,9,2
2005,198,355,62,9,4
2010,306,258,57,9,2
2015,330,231,8,59,4
2017,319,261,13,37,2


Numeric: ['previous_majority_proportion', 'Conservative', 'Labour', 'LD', 'previous_con_share', 'previous_lib_share', 'previous_lab_share', 'previous_natSW_share', 'projected_con_share', 'projected_lib_share', 'projected_lab_share']
Categorical: ['country/region', 'previous_winner', 'incumbent']
Training-derived class mapping: {0: 'con', 1: 'lab', 2: 'lib', 3: 'natSW', 4: 'oth'}


## Preprocessing and model

For each seed, fit a fresh ColumnTransformer on that seed's training rows only. Numeric predictors receive median imputation and standardisation (to aid neural-network optimisation). Categorical predictors receive a missing-value category and one-hot encoding with unknown categories ignored.

The network has hidden widths 64 and 32 with ReLU, and a raw-logit output for CrossEntropyLoss; there is no Softmax layer. Use Adam with learning rate 0.001. These choices stay fixed across seeds.

In [3]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_preprocessor():
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scale", StandardScaler()),
        ]), numeric_columns),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(
                strategy="constant", fill_value="__MISSING__", keep_empty_features=True
            )),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_columns),
    ])


def as_features(matrix):
    if sparse.issparse(matrix):
        matrix = matrix.toarray()
    array = np.asarray(matrix, dtype=np.float32)
    if not np.isfinite(array).all():
        raise ValueError("Preprocessed predictors contain non-finite values.")
    return torch.from_numpy(array).to(DEVICE)


def as_targets(series):
    return torch.tensor(
        label_encoder.transform(series), dtype=torch.long, device=DEVICE
    )


def make_model(input_width):
    return nn.Sequential(
        nn.Linear(input_width, HIDDEN_SIZES[0]),
        nn.ReLU(),
        nn.Linear(HIDDEN_SIZES[0], HIDDEN_SIZES[1]),
        nn.ReLU(),
        nn.Linear(HIDDEN_SIZES[1], len(label_encoder.classes_)),
    ).to(DEVICE)

## Train each seed and restore its best checkpoint

An improvement means loss is **at least 0.001 lower** than the best accepted loss. Save that epoch's weights, reset patience, and stop after 20 consecutive epochs without a qualifying improvement (or at 1500). Thus best_epoch refers to the accepted checkpoint, not the stopping epoch; a smaller reduction below min_delta does not replace it.

Validation loss is calculated across the entire validation set after each complete epoch. The 2019 features and labels are first transformed/accessed for evaluation after restoring the checkpoint. No 2019 score affects training.

In [4]:
def run_seed(seed):
    seed_everything(seed)
    counts = election_2017[TARGET].value_counts()
    smallest_half = len(election_2017) // 2
    can_stratify = counts.min() >= 2 and smallest_half >= len(counts)
    if not can_stratify:
        warnings.warn(f"Seed {seed}: class counts do not permit stratification.")
    # Randomise half of 2017 per seed, preserving party proportions where possible.
    train_2017, validation = train_test_split(
        election_2017,
        test_size=0.5,
        random_state=seed,
        stratify=election_2017[TARGET] if can_stratify else None,
    )
    training = pd.concat([historical, train_2017])
    assert set(training.index).isdisjoint(validation.index)
    assert set(training.index).isdisjoint(test_2019.index)
    assert set(validation.index).isdisjoint(test_2019.index)
    assert len(train_2017) + len(validation) == len(election_2017)

    preprocessor = make_preprocessor()
    x_train = as_features(preprocessor.fit_transform(training[FEATURES]))
    x_validation = as_features(preprocessor.transform(validation[FEATURES]))
    y_train = as_targets(training[TARGET])
    y_validation = as_targets(validation[TARGET])

    loader = DataLoader(
        TensorDataset(x_train, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
        num_workers=0,
    )
    model = make_model(x_train.shape[1])
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    best_loss = float("inf")
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for x_batch, y_batch in loader:
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            if not torch.isfinite(loss).item():
                raise RuntimeError(f"Non-finite training loss for seed {seed}, epoch {epoch}")
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            validation_loss = criterion(model(x_validation), y_validation).item()
        if not np.isfinite(validation_loss):
            raise RuntimeError(f"Non-finite validation loss for seed {seed}, epoch {epoch}")

        if best_loss - validation_loss >= MIN_DELTA:
            best_loss = validation_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            break

    stopping_epoch = epoch
    assert best_state is not None and 1 <= best_epoch <= stopping_epoch
    model.load_state_dict(best_state)
    model.eval()

    # First use of test predictors/targets: evaluation of the frozen checkpoint.
    x_test = as_features(preprocessor.transform(test_2019[FEATURES]))
    with torch.no_grad():
        predictions = model(x_test).argmax(dim=1)
        predicted_parties = label_encoder.inverse_transform(predictions.cpu().numpy())
        accuracy = float(np.mean(predicted_parties == test_2019[TARGET].to_numpy()))
        restored_loss = criterion(model(x_validation), y_validation).item()
    assert len(predictions) == len(test_2019)
    assert np.isclose(restored_loss, best_loss), "Checkpoint restoration failed."

    record = {
        "seed": seed,
        "best_epoch": best_epoch,
        "best_validation_loss": best_loss,
        "stopping_epoch": stopping_epoch,
        "2019_accuracy": accuracy,
    }
    # Retain every fitted checkpoint and its own preprocessing for later inspection.
    return record, model, preprocessor

In [5]:
# Run this cell to train the ten independent models and evaluate each on 2019.
records = []
trained_models = {}
for seed in SEEDS:
    result, model, preprocessor = run_seed(seed)
    records.append(result)
    trained_models[seed] = {"model": model, "preprocessor": preprocessor}
    print(
        f"Seed {seed}: best epoch {result['best_epoch']}, "
        f"total epochs {result['stopping_epoch']}, "
        f"validation loss {result['best_validation_loss']:.4f}, "
        f"2019 accuracy {result['2019_accuracy']:.2%}",
        flush=True,
    )

results = pd.DataFrame(records)[[
    "seed", "best_epoch", "stopping_epoch", "best_validation_loss", "2019_accuracy"
]]
assert len(results) == len(SEEDS) == results["seed"].nunique() == 10
assert results["2019_accuracy"].between(0, 1).all()
# Keep numeric values in results; format accuracy as a percentage for display.
display(results.style.format({"2019_accuracy": "{:.2%}", "best_validation_loss": "{:.4f}"}))

Seed 111223: best epoch 35, total epochs 55, validation loss 0.1852, 2019 accuracy 87.03%
Seed 111224: best epoch 38, total epochs 58, validation loss 0.1775, 2019 accuracy 87.50%
Seed 111225: best epoch 55, total epochs 75, validation loss 0.1944, 2019 accuracy 84.34%
Seed 111226: best epoch 50, total epochs 70, validation loss 0.2028, 2019 accuracy 86.23%
Seed 111227: best epoch 32, total epochs 52, validation loss 0.2180, 2019 accuracy 84.65%
Seed 111228: best epoch 49, total epochs 69, validation loss 0.1756, 2019 accuracy 88.92%
Seed 111229: best epoch 31, total epochs 51, validation loss 0.3125, 2019 accuracy 85.92%
Seed 111230: best epoch 43, total epochs 63, validation loss 0.2179, 2019 accuracy 85.60%
Seed 111231: best epoch 38, total epochs 58, validation loss 0.2291, 2019 accuracy 84.02%
Seed 111232: best epoch 33, total epochs 53, validation loss 0.2126, 2019 accuracy 83.86%


,seed,best_epoch,stopping_epoch,best_validation_loss,2019_accuracy
0,111223,35,55,0.1852,87.03%
1,111224,38,58,0.1775,87.50%
2,111225,55,75,0.1944,84.34%
3,111226,50,70,0.2028,86.23%
4,111227,32,52,0.2180,84.65%
5,111228,49,69,0.1756,88.92%
6,111229,31,51,0.3125,85.92%
7,111230,43,63,0.2179,85.60%
8,111231,38,58,0.2291,84.02%
9,111232,33,53,0.2126,83.86%


## Summarise stability and candidate training duration

Report sample standard deviations (ddof=1). Accuracy is shown as a percentage; its standard deviation is in percentage points. Use the median best epoch, not the epoch or seed with the highest 2019 accuracy. A fractional median can be rounded up for a later fixed-duration run.

These are descriptive results across splits/initialisations on the same test election, not uncertainty across future elections. The median is only a candidate: training on a larger dataset changes the number of updates per epoch. This notebook does not retrain a final model.

In [6]:
epochs = results["best_epoch"]
accuracies = results["2019_accuracy"]
median_best_epoch = epochs.median()
candidate_fixed_epochs = int(np.ceil(median_best_epoch))

print(f"Number of seeds evaluated: {len(results)}")
print(f"Median best epoch (main quantity): {median_best_epoch:.1f}")
print(f"Mean best epoch: {epochs.mean():.2f}")
print(f"Standard deviation of best epochs: {epochs.std(ddof=1):.2f}")
print(f"Minimum best epoch: {epochs.min()}")
print(f"Maximum best epoch: {epochs.max()}")
print(f"Mean 2019 accuracy: {accuracies.mean():.2%}")
print(f"Standard deviation of 2019 accuracy: {100 * accuracies.std(ddof=1):.2f} percentage points")
print(f"Minimum 2019 accuracy: {accuracies.min():.2%}")
print(f"Maximum 2019 accuracy: {accuracies.max():.2%}")
print(f"Candidate fixed epochs (median rounded up): {candidate_fixed_epochs}")

if results["stopping_epoch"].eq(MAX_EPOCHS).any():
    print("Some runs reached the epoch cap; inspect those runs before interpreting the duration.")

display(results.style.format({"2019_accuracy": "{:.2%}", "best_validation_loss": "{:.4f}"}))

Number of seeds evaluated: 10
Median best epoch (main quantity): 38.0
Mean best epoch: 40.40
Standard deviation of best epochs: 8.44
Minimum best epoch: 31
Maximum best epoch: 55
Mean 2019 accuracy: 85.81%
Standard deviation of 2019 accuracy: 1.66 percentage points
Minimum 2019 accuracy: 83.86%
Maximum 2019 accuracy: 88.92%
Candidate fixed epochs (median rounded up): 38


,seed,best_epoch,stopping_epoch,best_validation_loss,2019_accuracy
0,111223,35,55,0.1852,87.03%
1,111224,38,58,0.1775,87.50%
2,111225,55,75,0.1944,84.34%
3,111226,50,70,0.2028,86.23%
4,111227,32,52,0.2180,84.65%
5,111228,49,69,0.1756,88.92%
6,111229,31,51,0.3125,85.92%
7,111230,43,63,0.2179,85.60%
8,111231,38,58,0.2291,84.02%
9,111232,33,53,0.2126,83.86%


In [7]:
# Run after training all ten models. Preserve 2019 row and party-column order.
from scipy.special import softmax

if len(SEEDS) != 10 or set(trained_models) != set(SEEDS):
    raise ValueError("Train all ten models before running the ensemble evaluation.")

party_names = label_encoder.classes_
model_probabilities_2019 = {}
for seed in SEEDS:
    fitted = trained_models[seed]
    fitted["model"].eval()
    x_ensemble = as_features(fitted["preprocessor"].transform(test_2019[FEATURES]))
    with torch.no_grad():
        probabilities = torch.softmax(fitted["model"](x_ensemble), dim=1).cpu().numpy()
    model_probabilities_2019[seed] = pd.DataFrame(
        probabilities, index=test_2019.index, columns=party_names
    )

# Shape: (models, entries, parties); average only over the ten models.
probability_stack_2019 = np.stack([
    model_probabilities_2019[seed].to_numpy() for seed in SEEDS
])
assert np.isfinite(probability_stack_2019).all()
assert np.allclose(probability_stack_2019.sum(axis=2), 1.0)
mean_probabilities_2019 = pd.DataFrame(
    probability_stack_2019.mean(axis=0), index=test_2019.index, columns=party_names
)

# Means already sum to one. The requested extra softmax flattens probabilities
# but preserves the highest-probability party, so accuracy is unchanged.
ensemble_probabilities_2019 = pd.DataFrame(
    softmax(mean_probabilities_2019.to_numpy(), axis=1),
    index=test_2019.index, columns=party_names,
)
assert np.allclose(ensemble_probabilities_2019.sum(axis=1), 1.0)
ensemble_predictions_2019 = ensemble_probabilities_2019.idxmax(axis=1)
ensemble_results_2019 = pd.DataFrame({
    "actual_party": test_2019[TARGET],
    "predicted_party": ensemble_predictions_2019,
    "winning_probability": ensemble_probabilities_2019.max(axis=1),
})
ensemble_results_2019["correct"] = (
    ensemble_results_2019["predicted_party"] == ensemble_results_2019["actual_party"]
)
ensemble_accuracy_2019 = float(ensemble_results_2019["correct"].mean())
print(f"Ten-model ensemble 2019 accuracy: {ensemble_accuracy_2019:.2%}")
print(f"Correct predictions: {ensemble_results_2019['correct'].sum()} / {len(test_2019)}")
display(ensemble_results_2019.join(ensemble_probabilities_2019.add_prefix("probability_")))

# Match the pipeline's evaluation subset without retraining the ensemble.
missing_predictors_2019 = test_2019[FEATURES].isna().any(axis=1)
complete_results_2019 = ensemble_results_2019.loc[~missing_predictors_2019]
print(f"2019 rows with at least one missing predictor: {int(missing_predictors_2019.sum())} / {len(test_2019)}")
print(f"2019 rows retained after excluding missing predictors: {len(complete_results_2019)}")
if complete_results_2019.empty:
    ensemble_accuracy_2019_complete = float("nan")
    print("Ensemble accuracy excluding missing predictors: unavailable (no complete rows).")
else:
    ensemble_accuracy_2019_complete = float(complete_results_2019["correct"].mean())
    print(f"Ensemble 2019 accuracy excluding missing predictors: {ensemble_accuracy_2019_complete:.2%}")
    print(f"Correct predictions on complete rows: {int(complete_results_2019['correct'].sum())} / {len(complete_results_2019)}")


Ten-model ensemble 2019 accuracy: 86.39%
Correct predictions: 546 / 632


,actual_party,predicted_party,winning_probability,correct,probability_con,probability_lab,probability_lib,probability_natSW,probability_oth
3171,lab,lab,0.403373,True,0.149013,0.403373,0.149013,0.149019,0.149582
3172,con,con,0.356128,True,0.356128,0.180882,0.154177,0.154188,0.154625
3173,natSW,natSW,0.253003,True,0.245944,0.179485,0.160521,0.253003,0.161047
3174,natSW,con,0.394114,False,0.394114,0.151031,0.150189,0.154199,0.150466
3175,natSW,con,0.266433,False,0.266433,0.220446,0.160699,0.191189,0.161234
...,...,...,...,...,...,...,...,...,...
3798,lab,lab,0.404275,True,0.148932,0.404275,0.148893,0.148890,0.149010
3799,con,con,0.403722,True,0.403722,0.148961,0.149328,0.148962,0.149026
3800,con,lab,0.368468,False,0.170344,0.368468,0.153024,0.154476,0.153688
3801,lab,lab,0.404381,True,0.148894,0.404381,0.148880,0.148877,0.148968


2019 rows with at least one missing predictor: 3 / 632
2019 rows retained after excluding missing predictors: 629
Ensemble 2019 accuracy excluding missing predictors: 86.49%
Correct predictions on complete rows: 544 / 629
